In [12]:
pip install pandas numpy matplotlib seaborn pyecharts openpyxl

Note: you may need to restart the kernel to use updated packages.


In [13]:
pip install pandas numpy matplotlib seaborn pyecharts openpyxl

Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install --upgrade pyecharts

Requirement already up-to-date: pyecharts in c:\programdata\anaconda3\lib\site-packages (2.0.8)
Note: you may need to restart the kernel to use updated packages.


In [16]:
pip install pyecharts --upgrade

Requirement already up-to-date: pyecharts in c:\programdata\anaconda3\lib\site-packages (2.0.8)
Note: you may need to restart the kernel to use updated packages.


Retrying (Retry(total=4, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ProtocolError('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None))': /simple/pyecharts/


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import seaborn as sns
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Page, Grid
from pyecharts.components import Table
from pyecharts.globals import ThemeType
from pyecharts.commons.utils import JsCode
import openpyxl
import re
from jinja2 import Environment, FileSystemLoader

# 自定义模板环境，确保布尔值正确解析
def create_custom_environment():
    env = Environment(loader=FileSystemLoader("."))
    
    # 添加过滤器来正确处理布尔值
    def to_boolean(value):
        return "true" if value else "false"
    
    env.filters["to_boolean"] = to_boolean
    return env

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 读取数据
df = pd.read_excel('中国电影网电影_data.xlsx')

# 1. 数据预处理
# 清洗日期列，提取标准日期格式
def clean_date(date_str):
    # 使用正则表达式提取日期部分（格式如YYYY-MM-DD）
    match = re.search(r'(\d{4}-\d{1,2}-\d{1,2})', str(date_str))
    if match:
        return match.group(1)
    return np.nan

# 处理年份列，先清洗再提取年份
df['上映时间'] = df['上映时间'].apply(clean_date)
df['年份'] = pd.to_datetime(df['上映时间'], errors='coerce').dt.year

# 处理票房列，确保是数值类型
def clean_box_office(value):
    try:
        # 移除可能存在的逗号、货币符号等
        if isinstance(value, str):
            value = value.replace(',', '').replace('¥', '').replace('$', '').strip()
        return float(value)
    except (ValueError, TypeError):
        return np.nan

df['累计票房'] = df['累计票房'].apply(clean_box_office)

# 处理国家及地区列，取第一个国家作为主要国家
df['国家及地区'] = df['国家及地区'].str.split('/').str[0]

# 处理类型列，取第一个类型作为主要类型
df['类型'] = df['类型'].str.split('/').str[0]

# 检查并处理缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 删除关键列为NaN的行（无法确定年份或票房的数据）
df = df.dropna(subset=['年份', '累计票房'])
df['年份'] = df['年份'].astype(int)

# 2. 创建可视化大屏
# 使用自定义模板环境
custom_env = create_custom_environment()
page = Page(layout=Page.DraggablePageLayout, page_title="电影数据分析大屏")

# 辅助函数：设置图表通用属性
def setup_chart(chart, remove_br=False, page_interval=0):
    chart.remove_br = remove_br
    chart.page_interval = page_interval
    return chart

# (1) 中一：业绩总览
def create_overview():
    # 计算累计票房
    total_box_office = df['累计票房'].sum() / 10000  # 转换为亿元
    
    # 计算各年度票房占比
    box_office_by_year = df.groupby('年份')['累计票房'].sum()
    box_office_2018 = box_office_by_year.get(2018, 0) / box_office_by_year.sum() * 100
    box_office_2017 = box_office_by_year.get(2017, 0) / box_office_by_year.sum() * 100
    box_office_2016 = box_office_by_year.get(2016, 0) / box_office_by_year.sum() * 100
    
    # 创建卡片式展示
    overview = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="100%"))
        .add_xaxis(["业绩总览"])
        .add_yaxis("累计票房收入(亿元)", [total_box_office], 
                  label_opts=opts.LabelOpts(position="inside", formatter="{c}"))
        .set_global_opts(
            title_opts=opts.TitleOpts(title="业绩总览", subtitle=f"2018: {box_office_2018:.1f}% | 2017: {box_office_2017:.1f}% | 2016: {box_office_2016:.1f}%"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            tooltip_opts=opts.TooltipOpts(is_show=False),
            graphic_opts=[
                opts.GraphicText(
                    graphic_item=opts.GraphicItem(
                        left="center", top="40%"
                    ),
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text=f"{total_box_office:.2f}亿",
                        font="bold 36px Microsoft YaHei",
                        graphic_basicstyle_opts=opts.GraphicBasicStyleOpts(
                            fill="#fff"
                        )
                    )
                )
            ]
        )
    )
    return setup_chart(overview)

# (2) 左一：票房前十影片排行榜
def create_top10_movies():
    top10 = df.sort_values('累计票房', ascending=False).head(10)
    
    table = Table()
    headers = ["排名", "电影名", "票房(亿元)"]
    rows = []
    for i, (_, row) in enumerate(top10.iterrows(), 1):
        rows.append([str(i), row['电影名'], f"{row['累计票房']/10000:.2f}"])
    
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="票房前十影片排行榜")
    )
    return setup_chart(table)

# (3) 左二：各个国家影片数量的柱状图对比分析
def create_movies_by_country():
    country_counts = df['国家及地区'].value_counts().head(10)
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(country_counts.index.tolist())
        .add_yaxis("影片数量", country_counts.values.tolist())
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各国影片数量TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="数量"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    return setup_chart(bar)

# (4) 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
def create_yearly_box_office_and_count():
    yearly_stats = df.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_stats['累计票房'] = yearly_stats['累计票房'] / 10000  # 转换为亿元
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_stats['累计票房'].tolist(), yaxis_index=0)
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="年度票房与影片数量"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['电影名'].tolist(), yaxis_index=1)
    )
    
    bar.overlap(line)
    return setup_chart(bar)

# (5) 中三：各个电影公司的影片数量和票房总额（两个线性）
def create_company_stats():
    # 处理制作公司数据，取第一个公司作为主要公司
    df['发行公司'] = df['发行公司'].str.split('/').str[0]
    company_stats = df.groupby('发行公司').agg({'累计票房': 'sum', '电影名': 'count'}).sort_values('累计票房', ascending=False).head(10)
    company_stats['累计票房'] = company_stats['累计票房'] / 10000  # 转换为亿元
    
    line1 = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("票房总额(亿元)", company_stats['累计票房'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=company_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross")
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("影片数量", company_stats['电影名'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  yaxis_index=1,
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line1.overlap(line2)
    return setup_chart(line1)

# (6) 右一：纯国产电影各个年度的票房总额和数量走势
def create_domestic_movies_trend():
    domestic = df[df['国家及地区'] == '中国']
    yearly_domestic = domestic.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_domestic['累计票房'] = yearly_domestic['累计票房'] / 10000  # 转换为亿元
    
    line = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_domestic['累计票房'].tolist(), 
                  yaxis_index=0,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_domestic['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="国产电影年度趋势"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("影片数量", yearly_domestic['电影名'].tolist(), 
                  yaxis_index=1,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line.overlap(line2)
    return setup_chart(line)

# (7) 右二：各个年度各个影片类型的票房数据表
def create_genre_yearly_table():
    genre_yearly = df.groupby(['年份', '类型'])['累计票房'].sum().unstack().fillna(0)
    genre_yearly = genre_yearly / 10000  # 转换为亿元
    
    # 准备表格数据
    headers = ["年份"] + genre_yearly.columns.tolist()
    rows = []
    for year in genre_yearly.index:
        row = [str(year)] + [f"{v:.2f}" for v in genre_yearly.loc[year].values]
        rows.append(row)
    
    table = Table()
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="年度-类型票房数据(亿元)")
    )
    return setup_chart(table)

# (8) 右三：各个类型的电影数量饼图对比分析 - 修复空数据问题
def create_genre_pie():
    genre_counts = df['类型'].value_counts().head(10)
    
    # 确保有足够的数据
    if len(genre_counts) == 0:
        # 创建空饼图作为后备 - 使用虚拟数据
        pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        pie.add(
            "",
            [("无数据", 1)],
            radius=["30%", "70%"],
            center=["50%", "50%"],
            label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
        )
        pie.set_global_opts(
            title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="15%", pos_left="2%")
        )
        return setup_chart(pie)
    
    # 使用更安全的饼图创建方式
    pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
    
    # 添加数据
    data_pair = []
    for genre, count in genre_counts.items():
        data_pair.append(opts.PieItem(name=genre, value=count))
    
    pie.add(
        series_name="",
        data_pair=data_pair,
        radius=["30%", "70%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
    )
    
    # 设置全局选项
    pie.set_global_opts(
        title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_top="15%",
            pos_left="2%"
        )
    )
    return setup_chart(pie)

# 添加所有图表到页面
page.add(
    create_overview(),
    create_top10_movies(),
    create_movies_by_country(),
    create_yearly_box_office_and_count(),
    create_company_stats(),
    create_domestic_movies_trend(),
    create_genre_yearly_table(),
    create_genre_pie()
)

# 保存为HTML文件，使用自定义环境
page.render("电影数据分析大屏.html", env=custom_env)

print("数据大屏已生成，请查看'电影数据分析大屏.html'文件")

缺失值统计:
电影名         0
累计票房     3141
导演        175
主演        298
上映时间       45
国家及地区       5
发行公司      632
类型          7
链接          0
年份         45
dtype: int64


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:90: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:91: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:92: RuntimeWarning: invalid value encountered in double_scalars


TemplateNotFound: simple_page.html

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import seaborn as sns
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Page, Grid
from pyecharts.components import Table
from pyecharts.globals import ThemeType
from pyecharts.commons.utils import JsCode
import openpyxl
import re
import os
from jinja2 import Environment, FileSystemLoader

# 找到pyecharts模板目录
def get_pyecharts_templates_dir():
    import pyecharts
    return os.path.join(os.path.dirname(pyecharts.__file__), "templates")

# 自定义模板环境，确保布尔值正确解析并能找到pyecharts模板
def create_custom_environment():
    # 获取pyecharts模板目录
    templates_dir = get_pyecharts_templates_dir()
    # 创建加载器，从pyecharts模板目录加载
    loader = FileSystemLoader(templates_dir)
    env = Environment(loader=loader)
    
    # 添加过滤器来正确处理布尔值
    def to_boolean(value):
        return "true" if value else "false"
    
    env.filters["to_boolean"] = to_boolean
    return env

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 读取数据
df = pd.read_excel('中国电影网电影_data.xlsx')

# 1. 数据预处理
# 清洗日期列，提取标准日期格式
def clean_date(date_str):
    # 使用正则表达式提取日期部分（格式如YYYY-MM-DD）
    match = re.search(r'(\d{4}-\d{1,2}-\d{1,2})', str(date_str))
    if match:
        return match.group(1)
    return np.nan

# 处理年份列，先清洗再提取年份
df['上映时间'] = df['上映时间'].apply(clean_date)
df['年份'] = pd.to_datetime(df['上映时间'], errors='coerce').dt.year

# 处理票房列，确保是数值类型
def clean_box_office(value):
    try:
        # 移除可能存在的逗号、货币符号等
        if isinstance(value, str):
            value = value.replace(',', '').replace('¥', '').replace('$', '').strip()
        return float(value)
    except (ValueError, TypeError):
        return np.nan

df['累计票房'] = df['累计票房'].apply(clean_box_office)

# 处理国家及地区列，取第一个国家作为主要国家
df['国家及地区'] = df['国家及地区'].str.split('/').str[0]

# 处理类型列，取第一个类型作为主要类型
df['类型'] = df['类型'].str.split('/').str[0]

# 检查并处理缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 删除关键列为NaN的行（无法确定年份或票房的数据）
df = df.dropna(subset=['年份', '累计票房'])
df['年份'] = df['年份'].astype(int)

# 2. 创建可视化大屏
# 使用自定义模板环境
custom_env = create_custom_environment()
page = Page(layout=Page.DraggablePageLayout, page_title="电影数据分析大屏")

# 辅助函数：设置图表通用属性
def setup_chart(chart, remove_br=False, page_interval=0):
    chart.remove_br = remove_br
    chart.page_interval = page_interval
    return chart

# (1) 中一：业绩总览
def create_overview():
    # 计算累计票房
    total_box_office = df['累计票房'].sum() / 10000  # 转换为亿元
    
    # 计算各年度票房占比
    box_office_by_year = df.groupby('年份')['累计票房'].sum()
    box_office_2018 = box_office_by_year.get(2018, 0) / box_office_by_year.sum() * 100
    box_office_2017 = box_office_by_year.get(2017, 0) / box_office_by_year.sum() * 100
    box_office_2016 = box_office_by_year.get(2016, 0) / box_office_by_year.sum() * 100
    
    # 创建卡片式展示
    overview = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="100%"))
        .add_xaxis(["业绩总览"])
        .add_yaxis("累计票房收入(亿元)", [total_box_office], 
                  label_opts=opts.LabelOpts(position="inside", formatter="{c}"))
        .set_global_opts(
            title_opts=opts.TitleOpts(title="业绩总览", subtitle=f"2018: {box_office_2018:.1f}% | 2017: {box_office_2017:.1f}% | 2016: {box_office_2016:.1f}%"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            tooltip_opts=opts.TooltipOpts(is_show=False),
            graphic_opts=[
                opts.GraphicText(
                    graphic_item=opts.GraphicItem(
                        left="center", top="40%"
                    ),
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text=f"{total_box_office:.2f}亿",
                        font="bold 36px Microsoft YaHei",
                        graphic_basicstyle_opts=opts.GraphicBasicStyleOpts(
                            fill="#fff"
                        )
                    )
                )
            ]
        )
    )
    return setup_chart(overview)

# (2) 左一：票房前十影片排行榜
def create_top10_movies():
    top10 = df.sort_values('累计票房', ascending=False).head(10)
    
    table = Table()
    headers = ["排名", "电影名", "票房(亿元)"]
    rows = []
    for i, (_, row) in enumerate(top10.iterrows(), 1):
        rows.append([str(i), row['电影名'], f"{row['累计票房']/10000:.2f}"])
    
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="票房前十影片排行榜")
    )
    return setup_chart(table)

# (3) 左二：各个国家影片数量的柱状图对比分析
def create_movies_by_country():
    country_counts = df['国家及地区'].value_counts().head(10)
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(country_counts.index.tolist())
        .add_yaxis("影片数量", country_counts.values.tolist())
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各国影片数量TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="数量"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    return setup_chart(bar)

# (4) 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
def create_yearly_box_office_and_count():
    yearly_stats = df.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_stats['累计票房'] = yearly_stats['累计票房'] / 10000  # 转换为亿元
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_stats['累计票房'].tolist(), yaxis_index=0)
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="年度票房与影片数量"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['电影名'].tolist(), yaxis_index=1)
    )
    
    bar.overlap(line)
    return setup_chart(bar)

# (5) 中三：各个电影公司的影片数量和票房总额（两个线性）
def create_company_stats():
    # 处理制作公司数据，取第一个公司作为主要公司
    df['发行公司'] = df['发行公司'].str.split('/').str[0]
    company_stats = df.groupby('发行公司').agg({'累计票房': 'sum', '电影名': 'count'}).sort_values('累计票房', ascending=False).head(10)
    company_stats['累计票房'] = company_stats['累计票房'] / 10000  # 转换为亿元
    
    line1 = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("票房总额(亿元)", company_stats['累计票房'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=company_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross")
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("影片数量", company_stats['电影名'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  yaxis_index=1,
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line1.overlap(line2)
    return setup_chart(line1)

# (6) 右一：纯国产电影各个年度的票房总额和数量走势
def create_domestic_movies_trend():
    domestic = df[df['国家及地区'] == '中国']
    yearly_domestic = domestic.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_domestic['累计票房'] = yearly_domestic['累计票房'] / 10000  # 转换为亿元
    
    line = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_domestic['累计票房'].tolist(), 
                  yaxis_index=0,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_domestic['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="国产电影年度趋势"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("影片数量", yearly_domestic['电影名'].tolist(), 
                  yaxis_index=1,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line.overlap(line2)
    return setup_chart(line)

# (7) 右二：各个年度各个影片类型的票房数据表
def create_genre_yearly_table():
    genre_yearly = df.groupby(['年份', '类型'])['累计票房'].sum().unstack().fillna(0)
    genre_yearly = genre_yearly / 10000  # 转换为亿元
    
    # 准备表格数据
    headers = ["年份"] + genre_yearly.columns.tolist()
    rows = []
    for year in genre_yearly.index:
        row = [str(year)] + [f"{v:.2f}" for v in genre_yearly.loc[year].values]
        rows.append(row)
    
    table = Table()
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="年度-类型票房数据(亿元)")
    )
    return setup_chart(table)

# (8) 右三：各个类型的电影数量饼图对比分析 - 修复空数据问题
def create_genre_pie():
    genre_counts = df['类型'].value_counts().head(10)
    
    # 确保有足够的数据
    if len(genre_counts) == 0:
        # 创建空饼图作为后备 - 使用虚拟数据
        pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        pie.add(
            "",
            [("无数据", 1)],
            radius=["30%", "70%"],
            center=["50%", "50%"],
            label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
        )
        pie.set_global_opts(
            title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="15%", pos_left="2%")
        )
        return setup_chart(pie)
    
    # 使用更安全的饼图创建方式
    pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
    
    # 添加数据
    data_pair = []
    for genre, count in genre_counts.items():
        data_pair.append(opts.PieItem(name=genre, value=count))
    
    pie.add(
        series_name="",
        data_pair=data_pair,
        radius=["30%", "70%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
    )
    
    # 设置全局选项
    pie.set_global_opts(
        title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_top="15%",
            pos_left="2%"
        )
    )
    return setup_chart(pie)

# 添加所有图表到页面
page.add(
    create_overview(),
    create_top10_movies(),
    create_movies_by_country(),
    create_yearly_box_office_and_count(),
    create_company_stats(),
    create_domestic_movies_trend(),
    create_genre_yearly_table(),
    create_genre_pie()
)

# 保存为HTML文件，使用自定义环境
page.render("电影数据分析大屏.html", env=custom_env)

print("数据大屏已生成，请查看'电影数据分析大屏.html'文件")

缺失值统计:
电影名         0
累计票房     3141
导演        175
主演        298
上映时间       45
国家及地区       5
发行公司      632
类型          7
链接          0
年份         45
dtype: int64


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:100: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:101: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:102: RuntimeWarning: invalid value encountered in double_scalars


TemplateNotFound: simple_page.html

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import seaborn as sns
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Page, Grid
from pyecharts.components import Table
from pyecharts.globals import ThemeType
from pyecharts.commons.utils import JsCode
import openpyxl
import re
import os
from jinja2 import Environment, FileSystemLoader

# 可靠地获取pyecharts模板目录
def get_pyecharts_templates_path():
    """获取pyecharts安装目录下的templates文件夹路径"""
    import pyecharts
    # 方法1: 通过pyecharts.__file__获取安装路径
    pyecharts_path = os.path.dirname(pyecharts.__file__)
    templates_path = os.path.join(pyecharts_path, "templates")
    
    # 验证路径是否存在
    if not os.path.exists(templates_path):
        # 方法2: 备用方案，适用于某些安装情况
        import pkg_resources
        try:
            templates_path = pkg_resources.resource_filename("pyecharts", "templates")
        except:
            raise FileNotFoundError("无法找到pyecharts模板目录，请检查pyecharts是否正确安装")
    
    return templates_path

# 自定义模板环境，确保布尔值正确解析并能找到pyecharts模板
def create_custom_environment():
    """创建自定义Jinja2环境，正确加载pyecharts模板并处理布尔值"""
    # 获取pyecharts模板目录
    templates_dir = get_pyecharts_templates_path()
    # 创建加载器，从pyecharts模板目录加载
    loader = FileSystemLoader(templates_dir)
    env = Environment(loader=loader)
    
    # 添加过滤器来正确处理布尔值
    def to_boolean(value):
        """将Python布尔值转换为Jinja2可识别的布尔字符串"""
        return "true" if value else "false"
    
    env.filters["to_boolean"] = to_boolean
    return env

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 读取数据
df = pd.read_excel('中国电影网电影_data.xlsx')

# 1. 数据预处理
# 清洗日期列，提取标准日期格式
def clean_date(date_str):
    # 使用正则表达式提取日期部分（格式如YYYY-MM-DD）
    match = re.search(r'(\d{4}-\d{1,2}-\d{1,2})', str(date_str))
    if match:
        return match.group(1)
    return np.nan

# 处理年份列，先清洗再提取年份
df['上映时间'] = df['上映时间'].apply(clean_date)
df['年份'] = pd.to_datetime(df['上映时间'], errors='coerce').dt.year

# 处理票房列，确保是数值类型
def clean_box_office(value):
    try:
        # 移除可能存在的逗号、货币符号等
        if isinstance(value, str):
            value = value.replace(',', '').replace('¥', '').replace('$', '').strip()
        return float(value)
    except (ValueError, TypeError):
        return np.nan

df['累计票房'] = df['累计票房'].apply(clean_box_office)

# 处理国家及地区列，取第一个国家作为主要国家
df['国家及地区'] = df['国家及地区'].str.split('/').str[0]

# 处理类型列，取第一个类型作为主要类型
df['类型'] = df['类型'].str.split('/').str[0]

# 检查并处理缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 删除关键列为NaN的行（无法确定年份或票房的数据）
df = df.dropna(subset=['年份', '累计票房'])
df['年份'] = df['年份'].astype(int)

# 2. 创建可视化大屏
# 使用自定义模板环境
custom_env = create_custom_environment()
page = Page(layout=Page.DraggablePageLayout, page_title="电影数据分析大屏")

# 辅助函数：设置图表通用属性
def setup_chart(chart, remove_br=False, page_interval=0):
    """设置图表的通用属性，确保布尔值正确传递"""
    chart.remove_br = remove_br
    chart.page_interval = page_interval
    return chart

# (1) 中一：业绩总览
def create_overview():
    # 计算累计票房
    total_box_office = df['累计票房'].sum() / 10000  # 转换为亿元
    
    # 计算各年度票房占比
    box_office_by_year = df.groupby('年份')['累计票房'].sum()
    box_office_2018 = box_office_by_year.get(2018, 0) / box_office_by_year.sum() * 100
    box_office_2017 = box_office_by_year.get(2017, 0) / box_office_by_year.sum() * 100
    box_office_2016 = box_office_by_year.get(2016, 0) / box_office_by_year.sum() * 100
    
    # 创建卡片式展示
    overview = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="100%"))
        .add_xaxis(["业绩总览"])
        .add_yaxis("累计票房收入(亿元)", [total_box_office], 
                  label_opts=opts.LabelOpts(position="inside", formatter="{c}"))
        .set_global_opts(
            title_opts=opts.TitleOpts(title="业绩总览", subtitle=f"2018: {box_office_2018:.1f}% | 2017: {box_office_2017:.1f}% | 2016: {box_office_2016:.1f}%"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            tooltip_opts=opts.TooltipOpts(is_show=False),
            graphic_opts=[
                opts.GraphicText(
                    graphic_item=opts.GraphicItem(
                        left="center", top="40%"
                    ),
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text=f"{total_box_office:.2f}亿",
                        font="bold 36px Microsoft YaHei",
                        graphic_basicstyle_opts=opts.GraphicBasicStyleOpts(
                            fill="#fff"
                        )
                    )
                )
            ]
        )
    )
    return setup_chart(overview)

# (2) 左一：票房前十影片排行榜
def create_top10_movies():
    top10 = df.sort_values('累计票房', ascending=False).head(10)
    
    table = Table()
    headers = ["排名", "电影名", "票房(亿元)"]
    rows = []
    for i, (_, row) in enumerate(top10.iterrows(), 1):
        rows.append([str(i), row['电影名'], f"{row['累计票房']/10000:.2f}"])
    
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="票房前十影片排行榜")
    )
    return setup_chart(table)

# (3) 左二：各个国家影片数量的柱状图对比分析
def create_movies_by_country():
    country_counts = df['国家及地区'].value_counts().head(10)
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(country_counts.index.tolist())
        .add_yaxis("影片数量", country_counts.values.tolist())
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各国影片数量TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="数量"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    return setup_chart(bar)

# (4) 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
def create_yearly_box_office_and_count():
    yearly_stats = df.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_stats['累计票房'] = yearly_stats['累计票房'] / 10000  # 转换为亿元
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_stats['累计票房'].tolist(), yaxis_index=0)
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="年度票房与影片数量"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['电影名'].tolist(), yaxis_index=1)
    )
    
    bar.overlap(line)
    return setup_chart(bar)

# (5) 中三：各个电影公司的影片数量和票房总额（两个线性）
def create_company_stats():
    # 处理制作公司数据，取第一个公司作为主要公司
    df['发行公司'] = df['发行公司'].str.split('/').str[0]
    company_stats = df.groupby('发行公司').agg({'累计票房': 'sum', '电影名': 'count'}).sort_values('累计票房', ascending=False).head(10)
    company_stats['累计票房'] = company_stats['累计票房'] / 10000  # 转换为亿元
    
    line1 = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("票房总额(亿元)", company_stats['累计票房'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=company_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross")
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("影片数量", company_stats['电影名'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  yaxis_index=1,
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line1.overlap(line2)
    return setup_chart(line1)

# (6) 右一：纯国产电影各个年度的票房总额和数量走势
def create_domestic_movies_trend():
    domestic = df[df['国家及地区'] == '中国']
    yearly_domestic = domestic.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_domestic['累计票房'] = yearly_domestic['累计票房'] / 10000  # 转换为亿元
    
    line = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_domestic['累计票房'].tolist(), 
                  yaxis_index=0,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_domestic['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="国产电影年度趋势"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("影片数量", yearly_domestic['电影名'].tolist(), 
                  yaxis_index=1,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line.overlap(line2)
    return setup_chart(line)

# (7) 右二：各个年度各个影片类型的票房数据表
def create_genre_yearly_table():
    genre_yearly = df.groupby(['年份', '类型'])['累计票房'].sum().unstack().fillna(0)
    genre_yearly = genre_yearly / 10000  # 转换为亿元
    
    # 准备表格数据
    headers = ["年份"] + genre_yearly.columns.tolist()
    rows = []
    for year in genre_yearly.index:
        row = [str(year)] + [f"{v:.2f}" for v in genre_yearly.loc[year].values]
        rows.append(row)
    
    table = Table()
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="年度-类型票房数据(亿元)")
    )
    return setup_chart(table)

# (8) 右三：各个类型的电影数量饼图对比分析 - 修复空数据问题
def create_genre_pie():
    genre_counts = df['类型'].value_counts().head(10)
    
    # 确保有足够的数据
    if len(genre_counts) == 0:
        # 创建空饼图作为后备 - 使用虚拟数据
        pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        pie.add(
            "",
            [("无数据", 1)],
            radius=["30%", "70%"],
            center=["50%", "50%"],
            label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
        )
        pie.set_global_opts(
            title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="15%", pos_left="2%")
        )
        return setup_chart(pie)
    
    # 使用更安全的饼图创建方式
    pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
    
    # 添加数据
    data_pair = []
    for genre, count in genre_counts.items():
        data_pair.append(opts.PieItem(name=genre, value=count))
    
    pie.add(
        series_name="",
        data_pair=data_pair,
        radius=["30%", "70%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
    )
    
    # 设置全局选项
    pie.set_global_opts(
        title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_top="15%",
            pos_left="2%"
        )
    )
    return setup_chart(pie)

# 添加所有图表到页面
page.add(
    create_overview(),
    create_top10_movies(),
    create_movies_by_country(),
    create_yearly_box_office_and_count(),
    create_company_stats(),
    create_domestic_movies_trend(),
    create_genre_yearly_table(),
    create_genre_pie()
)

# 保存为HTML文件，使用自定义环境
page.render("电影数据分析大屏.html", env=custom_env)

print("数据大屏已生成，请查看'电影数据分析大屏.html'文件")

缺失值统计:
电影名         0
累计票房     3141
导演        175
主演        298
上映时间       45
国家及地区       5
发行公司      632
类型          7
链接          0
年份         45
dtype: int64


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:117: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:118: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:119: RuntimeWarning: invalid value encountered in double_scalars


TemplateNotFound: simple_page.html

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import seaborn as sns
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Page, Grid
from pyecharts.components import Table
from pyecharts.globals import ThemeType
from pyecharts.commons.utils import JsCode
import openpyxl
import re
import os
from jinja2 import Environment, FileSystemLoader, TemplateNotFound
import pyecharts.render.engine as engine

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 读取数据
df = pd.read_excel('中国电影网电影_data.xlsx')

# 1. 数据预处理
# 清洗日期列，提取标准日期格式
def clean_date(date_str):
    # 使用正则表达式提取日期部分（格式如YYYY-MM-DD）
    match = re.search(r'(\d{4}-\d{1,2}-\d{1,2})', str(date_str))
    if match:
        return match.group(1)
    return np.nan

# 处理年份列，先清洗再提取年份
df['上映时间'] = df['上映时间'].apply(clean_date)
df['年份'] = pd.to_datetime(df['上映时间'], errors='coerce').dt.year

# 处理票房列，确保是数值类型
def clean_box_office(value):
    try:
        # 移除可能存在的逗号、货币符号等
        if isinstance(value, str):
            value = value.replace(',', '').replace('¥', '').replace('$', '').strip()
        return float(value)
    except (ValueError, TypeError):
        return np.nan

df['累计票房'] = df['累计票房'].apply(clean_box_office)

# 处理国家及地区列，取第一个国家作为主要国家
df['国家及地区'] = df['国家及地区'].str.split('/').str[0]

# 处理类型列，取第一个类型作为主要类型
df['类型'] = df['类型'].str.split('/').str[0]

# 检查并处理缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 删除关键列为NaN的行（无法确定年份或票房的数据）
df = df.dropna(subset=['年份', '累计票房'])
df['年份'] = df['年份'].astype(int)

# 2. 修改pyecharts渲染引擎以正确处理布尔值
def custom_render_chart_to_template(self, template_name: str, chart: Any, **kwargs) -> str:
    """自定义渲染函数，处理布尔值"""
    try:
        tpl = self.env.get_template(template_name)
    except TemplateNotFound:
        # 尝试从pyecharts默认模板目录加载
        pyecharts_path = os.path.dirname(engine.__file__)
        templates_path = os.path.join(pyecharts_path, 'templates')
        loader = FileSystemLoader(templates_path)
        env = Environment(loader=loader)
        
        # 添加布尔值过滤器
        def to_boolean(value):
            return "true" if value else "false"
        
        env.filters["to_boolean"] = to_boolean
        tpl = env.get_template(template_name)
    
    # 确保布尔值正确传递
    chart_dict = chart.dump_options_with_quotes()
    # 递归处理字典中的布尔值
    def process_booleans(obj):
        if isinstance(obj, dict):
            return {k: process_booleans(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [process_booleans(v) for v in obj]
        elif isinstance(obj, bool):
            return "true" if obj else "false"
        return obj
    
    chart_dict = process_booleans(chart_dict)
    return tpl.render(chart=chart_dict, **kwargs)

# 替换pyecharts的渲染函数
engine.Engine.render_chart_to_template = custom_render_chart_to_template

# 3. 创建可视化大屏
page = Page(layout=Page.DraggablePageLayout, page_title="电影数据分析大屏")

# 辅助函数：设置图表通用属性
def setup_chart(chart, remove_br=False, page_interval=0):
    chart.options["remove_br"] = remove_br
    chart.options["page_interval"] = page_interval
    return chart

# (1) 中一：业绩总览
def create_overview():
    # 计算累计票房
    total_box_office = df['累计票房'].sum() / 10000  # 转换为亿元
    
    # 计算各年度票房占比
    box_office_by_year = df.groupby('年份')['累计票房'].sum()
    box_office_2018 = box_office_by_year.get(2018, 0) / box_office_by_year.sum() * 100
    box_office_2017 = box_office_by_year.get(2017, 0) / box_office_by_year.sum() * 100
    box_office_2016 = box_office_by_year.get(2016, 0) / box_office_by_year.sum() * 100
    
    # 创建卡片式展示
    overview = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="100%"))
        .add_xaxis(["业绩总览"])
        .add_yaxis("累计票房收入(亿元)", [total_box_office], 
                  label_opts=opts.LabelOpts(position="inside", formatter="{c}"))
        .set_global_opts(
            title_opts=opts.TitleOpts(title="业绩总览", subtitle=f"2018: {box_office_2018:.1f}% | 2017: {box_office_2017:.1f}% | 2016: {box_office_2016:.1f}%"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            tooltip_opts=opts.TooltipOpts(is_show=False),
            graphic_opts=[
                opts.GraphicText(
                    graphic_item=opts.GraphicItem(
                        left="center", top="40%"
                    ),
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text=f"{total_box_office:.2f}亿",
                        font="bold 36px Microsoft YaHei",
                        graphic_basicstyle_opts=opts.GraphicBasicStyleOpts(
                            fill="#fff"
                        )
                    )
                )
            ]
        )
    )
    return setup_chart(overview)

# (2) 左一：票房前十影片排行榜
def create_top10_movies():
    top10 = df.sort_values('累计票房', ascending=False).head(10)
    
    table = Table()
    headers = ["排名", "电影名", "票房(亿元)"]
    rows = []
    for i, (_, row) in enumerate(top10.iterrows(), 1):
        rows.append([str(i), row['电影名'], f"{row['累计票房']/10000:.2f}"])
    
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="票房前十影片排行榜")
    )
    return setup_chart(table)

# (3) 左二：各个国家影片数量的柱状图对比分析
def create_movies_by_country():
    country_counts = df['国家及地区'].value_counts().head(10)
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(country_counts.index.tolist())
        .add_yaxis("影片数量", country_counts.values.tolist())
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各国影片数量TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="数量"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    return setup_chart(bar)

# (4) 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
def create_yearly_box_office_and_count():
    yearly_stats = df.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_stats['累计票房'] = yearly_stats['累计票房'] / 10000  # 转换为亿元
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_stats['累计票房'].tolist(), yaxis_index=0)
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="年度票房与影片数量"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['电影名'].tolist(), yaxis_index=1)
    )
    
    bar.overlap(line)
    return setup_chart(bar)

# (5) 中三：各个电影公司的影片数量和票房总额（两个线性）
def create_company_stats():
    # 处理制作公司数据，取第一个公司作为主要公司
    df['发行公司'] = df['发行公司'].str.split('/').str[0]
    company_stats = df.groupby('发行公司').agg({'累计票房': 'sum', '电影名': 'count'}).sort_values('累计票房', ascending=False).head(10)
    company_stats['累计票房'] = company_stats['累计票房'] / 10000  # 转换为亿元
    
    line1 = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("票房总额(亿元)", company_stats['累计票房'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=company_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross")
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("影片数量", company_stats['电影名'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  yaxis_index=1,
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line1.overlap(line2)
    return setup_chart(line1)

# (6) 右一：纯国产电影各个年度的票房总额和数量走势
def create_domestic_movies_trend():
    domestic = df[df['国家及地区'] == '中国']
    yearly_domestic = domestic.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_domestic['累计票房'] = yearly_domestic['累计票房'] / 10000  # 转换为亿元
    
    line = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_domestic['累计票房'].tolist(), 
                  yaxis_index=0,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_domestic['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="国产电影年度趋势"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("影片数量", yearly_domestic['电影名'].tolist(), 
                  yaxis_index=1,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line.overlap(line2)
    return setup_chart(line)

# (7) 右二：各个年度各个影片类型的票房数据表
def create_genre_yearly_table():
    genre_yearly = df.groupby(['年份', '类型'])['累计票房'].sum().unstack().fillna(0)
    genre_yearly = genre_yearly / 10000  # 转换为亿元
    
    # 准备表格数据
    headers = ["年份"] + genre_yearly.columns.tolist()
    rows = []
    for year in genre_yearly.index:
        row = [str(year)] + [f"{v:.2f}" for v in genre_yearly.loc[year].values]
        rows.append(row)
    
    table = Table()
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="年度-类型票房数据(亿元)")
    )
    return setup_chart(table)

# (8) 右三：各个类型的电影数量饼图对比分析 - 修复空数据问题
def create_genre_pie():
    genre_counts = df['类型'].value_counts().head(10)
    
    # 确保有足够的数据
    if len(genre_counts) == 0:
        # 创建空饼图作为后备 - 使用虚拟数据
        pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        pie.add(
            "",
            [("无数据", 1)],
            radius=["30%", "70%"],
            center=["50%", "50%"],
            label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
        )
        pie.set_global_opts(
            title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="15%", pos_left="2%")
        )
        return setup_chart(pie)
    
    # 使用更安全的饼图创建方式
    pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
    
    # 添加数据
    data_pair = []
    for genre, count in genre_counts.items():
        data_pair.append(opts.PieItem(name=genre, value=count))
    
    pie.add(
        series_name="",
        data_pair=data_pair,
        radius=["30%", "70%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
    )
    
    # 设置全局选项
    pie.set_global_opts(
        title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_top="15%",
            pos_left="2%"
        )
    )
    return setup_chart(pie)

# 添加所有图表到页面
page.add(
    create_overview(),
    create_top10_movies(),
    create_movies_by_country(),
    create_yearly_box_office_and_count(),
    create_company_stats(),
    create_domestic_movies_trend(),
    create_genre_yearly_table(),
    create_genre_pie()
)

# 保存为HTML文件
page.render("电影数据分析大屏.html")

print("数据大屏已生成，请查看'电影数据分析大屏.html'文件")

缺失值统计:
电影名         0
累计票房     3141
导演        175
主演        298
上映时间       45
国家及地区       5
发行公司      632
类型          7
链接          0
年份         45
dtype: int64


NameError: name 'Any' is not defined

In [25]:
# 替换pyecharts的渲染函数 - 修正为TemplateRenderer
engine.TemplateRenderer.render_chart_to_template = custom_render_chart_to_template

AttributeError: module 'pyecharts.render.engine' has no attribute 'TemplateRenderer'

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import seaborn as sns
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Page, Grid
from pyecharts.components import Table
from pyecharts.globals import ThemeType
from pyecharts.commons.utils import JsCode
import openpyxl
import re
import os
from jinja2 import Environment, FileSystemLoader, TemplateNotFound
import pyecharts.render.engine as engine
from typing import Any

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 读取数据
df = pd.read_excel('中国电影网电影_data.xlsx')

# 1. 数据预处理
# 清洗日期列，提取标准日期格式
def clean_date(date_str):
    # 使用正则表达式提取日期部分（格式如YYYY-MM-DD）
    match = re.search(r'(\d{4}-\d{1,2}-\d{1,2})', str(date_str))
    if match:
        return match.group(1)
    return np.nan

# 处理年份列，先清洗再提取年份
df['上映时间'] = df['上映时间'].apply(clean_date)
df['年份'] = pd.to_datetime(df['上映时间'], errors='coerce').dt.year

# 处理票房列，确保是数值类型
def clean_box_office(value):
    try:
        # 移除可能存在的逗号、货币符号等
        if isinstance(value, str):
            value = value.replace(',', '').replace('¥', '').replace('$', '').strip()
        return float(value)
    except (ValueError, TypeError):
        return np.nan

df['累计票房'] = df['累计票房'].apply(clean_box_office)

# 处理国家及地区列，取第一个国家作为主要国家
df['国家及地区'] = df['国家及地区'].str.split('/').str[0]

# 处理类型列，取第一个类型作为主要类型
df['类型'] = df['类型'].str.split('/').str[0]

# 检查并处理缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 删除关键列为NaN的行（无法确定年份或票房的数据）
df = df.dropna(subset=['年份', '累计票房'])
df['年份'] = df['年份'].astype(int)

# 2. 修改pyecharts渲染引擎以正确处理布尔值
def custom_render_chart_to_template(self, template_name: str, chart: Any, **kwargs) -> str:
    """自定义渲染函数，处理布尔值"""
    try:
        tpl = self.env.get_template(template_name)
    except TemplateNotFound:
        # 尝试从pyecharts默认模板目录加载
        pyecharts_path = os.path.dirname(engine.__file__)
        templates_path = os.path.join(pyecharts_path, 'templates')
        loader = FileSystemLoader(templates_path)
        env = Environment(loader=loader)
        
        # 添加布尔值过滤器
        def to_boolean(value):
            return "true" if value else "false"
        
        env.filters["to_boolean"] = to_boolean
        tpl = env.get_template(template_name)
    
    # 确保布尔值正确传递
    chart_dict = chart.dump_options_with_quotes()
    # 递归处理字典中的布尔值
    def process_booleans(obj):
        if isinstance(obj, dict):
            return {k: process_booleans(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [process_booleans(v) for v in obj]
        elif isinstance(obj, bool):
            return "true" if obj else "false"
        return obj
    
    chart_dict = process_booleans(chart_dict)
    return tpl.render(chart=chart_dict, **kwargs)

# 替换pyecharts的渲染函数 - 修正为TemplateRenderer
engine.TemplateRenderer.render_chart_to_template = custom_render_chart_to_template

# 3. 创建可视化大屏
page = Page(layout=Page.DraggablePageLayout, page_title="电影数据分析大屏")

# 辅助函数：设置图表通用属性
def setup_chart(chart, remove_br=False, page_interval=0):
    chart.options["remove_br"] = remove_br
    chart.options["page_interval"] = page_interval
    return chart

# (1) 中一：业绩总览
def create_overview():
    # 计算累计票房
    total_box_office = df['累计票房'].sum() / 10000  # 转换为亿元
    
    # 计算各年度票房占比
    box_office_by_year = df.groupby('年份')['累计票房'].sum()
    box_office_2018 = box_office_by_year.get(2018, 0) / box_office_by_year.sum() * 100
    box_office_2017 = box_office_by_year.get(2017, 0) / box_office_by_year.sum() * 100
    box_office_2016 = box_office_by_year.get(2016, 0) / box_office_by_year.sum() * 100
    
    # 创建卡片式展示
    overview = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="100%"))
        .add_xaxis(["业绩总览"])
        .add_yaxis("累计票房收入(亿元)", [total_box_office], 
                  label_opts=opts.LabelOpts(position="inside", formatter="{c}"))
        .set_global_opts(
            title_opts=opts.TitleOpts(title="业绩总览", subtitle=f"2018: {box_office_2018:.1f}% | 2017: {box_office_2017:.1f}% | 2016: {box_office_2016:.1f}%"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            tooltip_opts=opts.TooltipOpts(is_show=False),
            graphic_opts=[
                opts.GraphicText(
                    graphic_item=opts.GraphicItem(
                        left="center", top="40%"
                    ),
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text=f"{total_box_office:.2f}亿",
                        font="bold 36px Microsoft YaHei",
                        graphic_basicstyle_opts=opts.GraphicBasicStyleOpts(
                            fill="#fff"
                        )
                    )
                )
            ]
        )
    )
    return setup_chart(overview)

# (2) 左一：票房前十影片排行榜
def create_top10_movies():
    top10 = df.sort_values('累计票房', ascending=False).head(10)
    
    table = Table()
    headers = ["排名", "电影名", "票房(亿元)"]
    rows = []
    for i, (_, row) in enumerate(top10.iterrows(), 1):
        rows.append([str(i), row['电影名'], f"{row['累计票房']/10000:.2f}"])
    
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="票房前十影片排行榜")
    )
    return setup_chart(table)

# (3) 左二：各个国家影片数量的柱状图对比分析
def create_movies_by_country():
    country_counts = df['国家及地区'].value_counts().head(10)
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(country_counts.index.tolist())
        .add_yaxis("影片数量", country_counts.values.tolist())
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各国影片数量TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="数量"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    return setup_chart(bar)

# (4) 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
def create_yearly_box_office_and_count():
    yearly_stats = df.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_stats['累计票房'] = yearly_stats['累计票房'] / 10000  # 转换为亿元
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_stats['累计票房'].tolist(), yaxis_index=0)
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="年度票房与影片数量"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['电影名'].tolist(), yaxis_index=1)
    )
    
    bar.overlap(line)
    return setup_chart(bar)

# (5) 中三：各个电影公司的影片数量和票房总额（两个线性）
def create_company_stats():
    # 处理制作公司数据，取第一个公司作为主要公司
    df['发行公司'] = df['发行公司'].str.split('/').str[0]
    company_stats = df.groupby('发行公司').agg({'累计票房': 'sum', '电影名': 'count'}).sort_values('累计票房', ascending=False).head(10)
    company_stats['累计票房'] = company_stats['累计票房'] / 10000  # 转换为亿元
    
    line1 = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("票房总额(亿元)", company_stats['累计票房'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=company_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross")
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("影片数量", company_stats['电影名'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  yaxis_index=1,
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line1.overlap(line2)
    return setup_chart(line1)

# (6) 右一：纯国产电影各个年度的票房总额和数量走势
def create_domestic_movies_trend():
    domestic = df[df['国家及地区'] == '中国']
    yearly_domestic = domestic.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_domestic['累计票房'] = yearly_domestic['累计票房'] / 10000  # 转换为亿元
    
    line = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_domestic['累计票房'].tolist(), 
                  yaxis_index=0,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_domestic['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="国产电影年度趋势"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("影片数量", yearly_domestic['电影名'].tolist(), 
                  yaxis_index=1,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line.overlap(line2)
    return setup_chart(line)

# (7) 右二：各个年度各个影片类型的票房数据表
def create_genre_yearly_table():
    genre_yearly = df.groupby(['年份', '类型'])['累计票房'].sum().unstack().fillna(0)
    genre_yearly = genre_yearly / 10000  # 转换为亿元
    
    # 准备表格数据
    headers = ["年份"] + genre_yearly.columns.tolist()
    rows = []
    for year in genre_yearly.index:
        row = [str(year)] + [f"{v:.2f}" for v in genre_yearly.loc[year].values]
        rows.append(row)
    
    table = Table()
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="年度-类型票房数据(亿元)")
    )
    return setup_chart(table)

# (8) 右三：各个类型的电影数量饼图对比分析 - 修复空数据问题
def create_genre_pie():
    genre_counts = df['类型'].value_counts().head(10)
    
    # 确保有足够的数据
    if len(genre_counts) == 0:
        # 创建空饼图作为后备 - 使用虚拟数据
        pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        pie.add(
            "",
            [("无数据", 1)],
            radius=["30%", "70%"],
            center=["50%", "50%"],
            label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
        )
        pie.set_global_opts(
            title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="15%", pos_left="2%")
        )
        return setup_chart(pie)
    
    # 使用更安全的饼图创建方式
    pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
    
    # 添加数据
    data_pair = []
    for genre, count in genre_counts.items():
        data_pair.append(opts.PieItem(name=genre, value=count))
    
    pie.add(
        series_name="",
        data_pair=data_pair,
        radius=["30%", "70%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
    )
    
    # 设置全局选项
    pie.set_global_opts(
        title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_top="15%",
            pos_left="2%"
        )
    )
    return setup_chart(pie)

# 添加所有图表到页面
page.add(
    create_overview(),
    create_top10_movies(),
    create_movies_by_country(),
    create_yearly_box_office_and_count(),
    create_company_stats(),
    create_domestic_movies_trend(),
    create_genre_yearly_table(),
    create_genre_pie()
)

# 保存为HTML文件
page.render("电影数据分析大屏.html")

print("数据大屏已生成，请查看'电影数据分析大屏.html'文件")

缺失值统计:
电影名         0
累计票房     3141
导演        175
主演        298
上映时间       45
国家及地区       5
发行公司      632
类型          7
链接          0
年份         45
dtype: int64


AttributeError: module 'pyecharts.render.engine' has no attribute 'TemplateRenderer'

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import seaborn as sns
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Page, Grid
from pyecharts.components import Table
from pyecharts.globals import ThemeType
from pyecharts.commons.utils import JsCode
import openpyxl
import re
import os
from jinja2 import Environment, FileSystemLoader, TemplateNotFound
import pyecharts.render.engine as engine
from typing import Any

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 读取数据
df = pd.read_excel('中国电影网电影_data.xlsx')

# 1. 数据预处理
# 清洗日期列，提取标准日期格式
def clean_date(date_str):
    # 使用正则表达式提取日期部分（格式如YYYY-MM-DD）
    match = re.search(r'(\d{4}-\d{1,2}-\d{1,2})', str(date_str))
    if match:
        return match.group(1)
    return np.nan

# 处理年份列，先清洗再提取年份
df['上映时间'] = df['上映时间'].apply(clean_date)
df['年份'] = pd.to_datetime(df['上映时间'], errors='coerce').dt.year

# 处理票房列，确保是数值类型
def clean_box_office(value):
    try:
        # 移除可能存在的逗号、货币符号等
        if isinstance(value, str):
            value = value.replace(',', '').replace('¥', '').replace('$', '').strip()
        return float(value)
    except (ValueError, TypeError):
        return np.nan

df['累计票房'] = df['累计票房'].apply(clean_box_office)

# 处理国家及地区列，取第一个国家作为主要国家
df['国家及地区'] = df['国家及地区'].str.split('/').str[0]

# 处理类型列，取第一个类型作为主要类型
df['类型'] = df['类型'].str.split('/').str[0]

# 检查并处理缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 删除关键列为NaN的行（无法确定年份或票房的数据）
df = df.dropna(subset=['年份', '累计票房'])
df['年份'] = df['年份'].astype(int)

# 2. 修改pyecharts渲染引擎以正确处理布尔值
def custom_render_chart_to_template(self, template_name: str, chart: Any, **kwargs) -> str:
    """自定义渲染函数，处理布尔值"""
    try:
        tpl = self.env.get_template(template_name)
    except TemplateNotFound:
        # 尝试从pyecharts默认模板目录加载
        pyecharts_path = os.path.dirname(engine.__file__)
        templates_path = os.path.join(pyecharts_path, 'templates')
        loader = FileSystemLoader(templates_path)
        env = Environment(loader=loader)
        
        # 添加布尔值过滤器
        def to_boolean(value):
            return "true" if value else "false"
        
        env.filters["to_boolean"] = to_boolean
        tpl = env.get_template(template_name)
    
    # 确保布尔值正确传递
    chart_dict = chart.dump_options_with_quotes()
    # 递归处理字典中的布尔值
    def process_booleans(obj):
        if isinstance(obj, dict):
            return {k: process_booleans(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [process_booleans(v) for v in obj]
        elif isinstance(obj, bool):
            return "true" if obj else "false"
        return obj
    
    chart_dict = process_booleans(chart_dict)
    return tpl.render(chart=chart_dict, **kwargs)

# 3. 动态适配不同版本的pyecharts渲染引擎
def patch_pyecharts_renderer():
    """动态检测并修补pyecharts渲染引擎"""
    # 尝试检测可用的渲染器类
    renderer_class = None
    
    # 尝试新版本的TemplateRenderer
    if hasattr(engine, 'TemplateRenderer'):
        renderer_class = engine.TemplateRenderer
    # 尝试旧版本的Engine
    elif hasattr(engine, 'Engine'):
        renderer_class = engine.Engine
    # 尝试其他可能的类名
    elif hasattr(engine, '_EChartsRenderer'):
        renderer_class = engine._EChartsRenderer
    
    if renderer_class:
        # 替换渲染方法
        renderer_class.render_chart_to_template = custom_render_chart_to_template
        print(f"已成功修补pyecharts渲染引擎: {renderer_class.__name__}")
    else:
        # 无法找到合适的渲染器类，使用备用方案
        print("警告: 无法自动检测pyecharts渲染引擎类，尝试使用默认方式...")
        # 创建一个基本的渲染器
        class BasicRenderer:
            def __init__(self):
                pyecharts_path = os.path.dirname(engine.__file__)
                templates_path = os.path.join(pyecharts_path, 'templates')
                self.env = Environment(loader=FileSystemLoader(templates_path))
            
            def render_chart_to_template(self, template_name, chart, **kwargs):
                return custom_render_chart_to_template(self, template_name, chart, **kwargs)
        
        # 使用基本渲染器
        page._renderer = BasicRenderer()
        print("已使用备用渲染器")

# 应用渲染引擎补丁
patch_pyecharts_renderer()

# 4. 创建可视化大屏
page = Page(layout=Page.DraggablePageLayout, page_title="电影数据分析大屏")

# 辅助函数：设置图表通用属性
def setup_chart(chart, remove_br=False, page_interval=0):
    chart.options["remove_br"] = remove_br
    chart.options["page_interval"] = page_interval
    return chart

# (1) 中一：业绩总览
def create_overview():
    # 计算累计票房
    total_box_office = df['累计票房'].sum() / 10000  # 转换为亿元
    
    # 计算各年度票房占比
    box_office_by_year = df.groupby('年份')['累计票房'].sum()
    box_office_2018 = box_office_by_year.get(2018, 0) / box_office_by_year.sum() * 100
    box_office_2017 = box_office_by_year.get(2017, 0) / box_office_by_year.sum() * 100
    box_office_2016 = box_office_by_year.get(2016, 0) / box_office_by_year.sum() * 100
    
    # 创建卡片式展示
    overview = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="100%"))
        .add_xaxis(["业绩总览"])
        .add_yaxis("累计票房收入(亿元)", [total_box_office], 
                  label_opts=opts.LabelOpts(position="inside", formatter="{c}"))
        .set_global_opts(
            title_opts=opts.TitleOpts(title="业绩总览", subtitle=f"2018: {box_office_2018:.1f}% | 2017: {box_office_2017:.1f}% | 2016: {box_office_2016:.1f}%"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            tooltip_opts=opts.TooltipOpts(is_show=False),
            graphic_opts=[
                opts.GraphicText(
                    graphic_item=opts.GraphicItem(
                        left="center", top="40%"
                    ),
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text=f"{total_box_office:.2f}亿",
                        font="bold 36px Microsoft YaHei",
                        graphic_basicstyle_opts=opts.GraphicBasicStyleOpts(
                            fill="#fff"
                        )
                    )
                )
            ]
        )
    )
    return setup_chart(overview)

# (2) 左一：票房前十影片排行榜
def create_top10_movies():
    top10 = df.sort_values('累计票房', ascending=False).head(10)
    
    table = Table()
    headers = ["排名", "电影名", "票房(亿元)"]
    rows = []
    for i, (_, row) in enumerate(top10.iterrows(), 1):
        rows.append([str(i), row['电影名'], f"{row['累计票房']/10000:.2f}"])
    
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="票房前十影片排行榜")
    )
    return setup_chart(table)

# (3) 左二：各个国家影片数量的柱状图对比分析
def create_movies_by_country():
    country_counts = df['国家及地区'].value_counts().head(10)
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(country_counts.index.tolist())
        .add_yaxis("影片数量", country_counts.values.tolist())
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各国影片数量TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="数量"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    return setup_chart(bar)

# (4) 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
def create_yearly_box_office_and_count():
    yearly_stats = df.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_stats['累计票房'] = yearly_stats['累计票房'] / 10000  # 转换为亿元
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_stats['累计票房'].tolist(), yaxis_index=0)
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="年度票房与影片数量"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['电影名'].tolist(), yaxis_index=1)
    )
    
    bar.overlap(line)
    return setup_chart(bar)

# (5) 中三：各个电影公司的影片数量和票房总额（两个线性）
def create_company_stats():
    # 处理制作公司数据，取第一个公司作为主要公司
    df['发行公司'] = df['发行公司'].str.split('/').str[0]
    company_stats = df.groupby('发行公司').agg({'累计票房': 'sum', '电影名': 'count'}).sort_values('累计票房', ascending=False).head(10)
    company_stats['累计票房'] = company_stats['累计票房'] / 10000  # 转换为亿元
    
    line1 = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("票房总额(亿元)", company_stats['累计票房'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=company_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross")
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("影片数量", company_stats['电影名'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  yaxis_index=1,
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line1.overlap(line2)
    return setup_chart(line1)

# (6) 右一：纯国产电影各个年度的票房总额和数量走势
def create_domestic_movies_trend():
    domestic = df[df['国家及地区'] == '中国']
    yearly_domestic = domestic.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_domestic['累计票房'] = yearly_domestic['累计票房'] / 10000  # 转换为亿元
    
    line = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_domestic['累计票房'].tolist(), 
                  yaxis_index=0,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_domestic['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="国产电影年度趋势"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("影片数量", yearly_domestic['电影名'].tolist(), 
                  yaxis_index=1,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line.overlap(line2)
    return setup_chart(line)

# (7) 右二：各个年度各个影片类型的票房数据表
def create_genre_yearly_table():
    genre_yearly = df.groupby(['年份', '类型'])['累计票房'].sum().unstack().fillna(0)
    genre_yearly = genre_yearly / 10000  # 转换为亿元
    
    # 准备表格数据
    headers = ["年份"] + genre_yearly.columns.tolist()
    rows = []
    for year in genre_yearly.index:
        row = [str(year)] + [f"{v:.2f}" for v in genre_yearly.loc[year].values]
        rows.append(row)
    
    table = Table()
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="年度-类型票房数据(亿元)")
    )
    return setup_chart(table)

# (8) 右三：各个类型的电影数量饼图对比分析 - 修复空数据问题
def create_genre_pie():
    genre_counts = df['类型'].value_counts().head(10)
    
    # 确保有足够的数据
    if len(genre_counts) == 0:
        # 创建空饼图作为后备 - 使用虚拟数据
        pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        pie.add(
            "",
            [("无数据", 1)],
            radius=["30%", "70%"],
            center=["50%", "50%"],
            label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
        )
        pie.set_global_opts(
            title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="15%", pos_left="2%")
        )
        return setup_chart(pie)
    
    # 使用更安全的饼图创建方式
    pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
    
    # 添加数据
    data_pair = []
    for genre, count in genre_counts.items():
        data_pair.append(opts.PieItem(name=genre, value=count))
    
    pie.add(
        series_name="",
        data_pair=data_pair,
        radius=["30%", "70%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
    )
    
    # 设置全局选项
    pie.set_global_opts(
        title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_top="15%",
            pos_left="2%"
        )
    )
    return setup_chart(pie)

# 添加所有图表到页面
page.add(
    create_overview(),
    create_top10_movies(),
    create_movies_by_country(),
    create_yearly_box_office_and_count(),
    create_company_stats(),
    create_domestic_movies_trend(),
    create_genre_yearly_table(),
    create_genre_pie()
)

# 保存为HTML文件
page.render("电影数据分析大屏.html")

print("数据大屏已生成，请查看'电影数据分析大屏.html'文件")

缺失值统计:
电影名         0
累计票房     3141
导演        175
主演        298
上映时间       45
国家及地区       5
发行公司      632
类型          7
链接          0
年份         45
dtype: int64
警告: 无法自动检测pyecharts渲染引擎类，尝试使用默认方式...
已使用备用渲染器


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:154: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:155: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:156: RuntimeWarning: invalid value encountered in double_scalars


AttributeError: 'Table' object has no attribute 'options'

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import seaborn as sns
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Page, Grid
from pyecharts.components import Table
from pyecharts.globals import ThemeType
from pyecharts.commons.utils import JsCode
import openpyxl
import re
import os
from jinja2 import Environment, FileSystemLoader, TemplateNotFound
import pyecharts.render.engine as engine
from typing import Any

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 读取数据
df = pd.read_excel('中国电影网电影_data.xlsx')

# 1. 数据预处理
# 清洗日期列，提取标准日期格式
def clean_date(date_str):
    # 使用正则表达式提取日期部分（格式如YYYY-MM-DD）
    match = re.search(r'(\d{4}-\d{1,2}-\d{1,2})', str(date_str))
    if match:
        return match.group(1)
    return np.nan

# 处理年份列，先清洗再提取年份
df['上映时间'] = df['上映时间'].apply(clean_date)
df['年份'] = pd.to_datetime(df['上映时间'], errors='coerce').dt.year

# 处理票房列，确保是数值类型
def clean_box_office(value):
    try:
        # 移除可能存在的逗号、货币符号等
        if isinstance(value, str):
            value = value.replace(',', '').replace('¥', '').replace('$', '').strip()
        return float(value)
    except (ValueError, TypeError):
        return np.nan

df['累计票房'] = df['累计票房'].apply(clean_box_office)

# 处理国家及地区列，取第一个国家作为主要国家
df['国家及地区'] = df['国家及地区'].str.split('/').str[0]

# 处理类型列，取第一个类型作为主要类型
df['类型'] = df['类型'].str.split('/').str[0]

# 检查并处理缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 删除关键列为NaN的行（无法确定年份或票房的数据）
df = df.dropna(subset=['年份', '累计票房'])
df['年份'] = df['年份'].astype(int)

# 2. 修改pyecharts渲染引擎以正确处理布尔值
def custom_render_chart_to_template(self, template_name: str, chart: Any, **kwargs) -> str:
    """自定义渲染函数，处理布尔值"""
    try:
        tpl = self.env.get_template(template_name)
    except TemplateNotFound:
        # 尝试从pyecharts默认模板目录加载
        pyecharts_path = os.path.dirname(engine.__file__)
        templates_path = os.path.join(pyecharts_path, 'templates')
        loader = FileSystemLoader(templates_path)
        env = Environment(loader=loader)
        
        # 添加布尔值过滤器
        def to_boolean(value):
            return "true" if value else "false"
        
        env.filters["to_boolean"] = to_boolean
        tpl = env.get_template(template_name)
    
    # 确保布尔值正确传递
    chart_dict = chart.dump_options_with_quotes()
    # 递归处理字典中的布尔值
    def process_booleans(obj):
        if isinstance(obj, dict):
            return {k: process_booleans(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [process_booleans(v) for v in obj]
        elif isinstance(obj, bool):
            return "true" if obj else "false"
        return obj
    
    chart_dict = process_booleans(chart_dict)
    return tpl.render(chart=chart_dict, **kwargs)

# 3. 动态适配不同版本的pyecharts渲染引擎
def patch_pyecharts_renderer():
    """动态检测并修补pyecharts渲染引擎"""
    # 尝试检测可用的渲染器类
    renderer_class = None
    
    # 尝试新版本的TemplateRenderer
    if hasattr(engine, 'TemplateRenderer'):
        renderer_class = engine.TemplateRenderer
    # 尝试旧版本的Engine
    elif hasattr(engine, 'Engine'):
        renderer_class = engine.Engine
    # 尝试其他可能的类名
    elif hasattr(engine, '_EChartsRenderer'):
        renderer_class = engine._EChartsRenderer
    
    if renderer_class:
        # 替换渲染方法
        renderer_class.render_chart_to_template = custom_render_chart_to_template
        print(f"已成功修补pyecharts渲染引擎: {renderer_class.__name__}")
    else:
        # 无法找到合适的渲染器类，使用备用方案
        print("警告: 无法自动检测pyecharts渲染引擎类，尝试使用默认方式...")
        # 创建一个基本的渲染器
        class BasicRenderer:
            def __init__(self):
                pyecharts_path = os.path.dirname(engine.__file__)
                templates_path = os.path.join(pyecharts_path, 'templates')
                self.env = Environment(loader=FileSystemLoader(templates_path))
            
            def render_chart_to_template(self, template_name, chart, **kwargs):
                return custom_render_chart_to_template(self, template_name, chart, **kwargs)
        
        # 使用基本渲染器
        page._renderer = BasicRenderer()
        print("已使用备用渲染器")

# 应用渲染引擎补丁
patch_pyecharts_renderer()

# 4. 创建可视化大屏
page = Page(layout=Page.DraggablePageLayout, page_title="电影数据分析大屏")

# 辅助函数：设置图表通用属性 - 修复Table组件不支持options的问题
def setup_chart(chart, remove_br=False, page_interval=0):
    # 检查是否为Table组件
    if isinstance(chart, Table):
        # Table组件使用set_global_opts设置选项
        chart.set_global_opts(
            toolbox_opts=opts.ToolboxOpts(is_show=True),
            title_opts=chart.options.get("title_opts", None)  # 保留原有标题
        )
    else:
        # 其他图表组件使用options属性
        chart.options["remove_br"] = remove_br
        chart.options["page_interval"] = page_interval
    return chart

# (1) 中一：业绩总览
def create_overview():
    # 计算累计票房
    total_box_office = df['累计票房'].sum() / 10000  # 转换为亿元
    
    # 计算各年度票房占比
    box_office_by_year = df.groupby('年份')['累计票房'].sum()
    box_office_2018 = box_office_by_year.get(2018, 0) / box_office_by_year.sum() * 100
    box_office_2017 = box_office_by_year.get(2017, 0) / box_office_by_year.sum() * 100
    box_office_2016 = box_office_by_year.get(2016, 0) / box_office_by_year.sum() * 100
    
    # 创建卡片式展示
    overview = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="100%"))
        .add_xaxis(["业绩总览"])
        .add_yaxis("累计票房收入(亿元)", [total_box_office], 
                  label_opts=opts.LabelOpts(position="inside", formatter="{c}"))
        .set_global_opts(
            title_opts=opts.TitleOpts(title="业绩总览", subtitle=f"2018: {box_office_2018:.1f}% | 2017: {box_office_2017:.1f}% | 2016: {box_office_2016:.1f}%"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            tooltip_opts=opts.TooltipOpts(is_show=False),
            graphic_opts=[
                opts.GraphicText(
                    graphic_item=opts.GraphicItem(
                        left="center", top="40%"
                    ),
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text=f"{total_box_office:.2f}亿",
                        font="bold 36px Microsoft YaHei",
                        graphic_basicstyle_opts=opts.GraphicBasicStyleOpts(
                            fill="#fff"
                        )
                    )
                )
            ]
        )
    )
    return setup_chart(overview)

# (2) 左一：票房前十影片排行榜
def create_top10_movies():
    top10 = df.sort_values('累计票房', ascending=False).head(10)
    
    table = Table()
    headers = ["排名", "电影名", "票房(亿元)"]
    rows = []
    for i, (_, row) in enumerate(top10.iterrows(), 1):
        rows.append([str(i), row['电影名'], f"{row['累计票房']/10000:.2f}"])
    
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="票房前十影片排行榜")
    )
    return setup_chart(table)

# (3) 左二：各个国家影片数量的柱状图对比分析
def create_movies_by_country():
    country_counts = df['国家及地区'].value_counts().head(10)
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(country_counts.index.tolist())
        .add_yaxis("影片数量", country_counts.values.tolist())
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各国影片数量TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="数量"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    return setup_chart(bar)

# (4) 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
def create_yearly_box_office_and_count():
    yearly_stats = df.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_stats['累计票房'] = yearly_stats['累计票房'] / 10000  # 转换为亿元
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_stats['累计票房'].tolist(), yaxis_index=0)
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="年度票房与影片数量"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['电影名'].tolist(), yaxis_index=1)
    )
    
    bar.overlap(line)
    return setup_chart(bar)

# (5) 中三：各个电影公司的影片数量和票房总额（两个线性）
def create_company_stats():
    # 处理制作公司数据，取第一个公司作为主要公司
    df['发行公司'] = df['发行公司'].str.split('/').str[0]
    company_stats = df.groupby('发行公司').agg({'累计票房': 'sum', '电影名': 'count'}).sort_values('累计票房', ascending=False).head(10)
    company_stats['累计票房'] = company_stats['累计票房'] / 10000  # 转换为亿元
    
    line1 = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("票房总额(亿元)", company_stats['累计票房'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=company_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross")
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("影片数量", company_stats['电影名'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  yaxis_index=1,
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line1.overlap(line2)
    return setup_chart(line1)

# (6) 右一：纯国产电影各个年度的票房总额和数量走势
def create_domestic_movies_trend():
    domestic = df[df['国家及地区'] == '中国']
    yearly_domestic = domestic.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_domestic['累计票房'] = yearly_domestic['累计票房'] / 10000  # 转换为亿元
    
    line = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_domestic['累计票房'].tolist(), 
                  yaxis_index=0,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_domestic['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="国产电影年度趋势"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("影片数量", yearly_domestic['电影名'].tolist(), 
                  yaxis_index=1,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line.overlap(line2)
    return setup_chart(line)

# (7) 右二：各个年度各个影片类型的票房数据表
def create_genre_yearly_table():
    genre_yearly = df.groupby(['年份', '类型'])['累计票房'].sum().unstack().fillna(0)
    genre_yearly = genre_yearly / 10000  # 转换为亿元
    
    # 准备表格数据
    headers = ["年份"] + genre_yearly.columns.tolist()
    rows = []
    for year in genre_yearly.index:
        row = [str(year)] + [f"{v:.2f}" for v in genre_yearly.loc[year].values]
        rows.append(row)
    
    table = Table()
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="年度-类型票房数据(亿元)")
    )
    return setup_chart(table)

# (8) 右三：各个类型的电影数量饼图对比分析 - 修复空数据问题
def create_genre_pie():
    genre_counts = df['类型'].value_counts().head(10)
    
    # 确保有足够的数据
    if len(genre_counts) == 0:
        # 创建空饼图作为后备 - 使用虚拟数据
        pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        pie.add(
            "",
            [("无数据", 1)],
            radius=["30%", "70%"],
            center=["50%", "50%"],
            label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
        )
        pie.set_global_opts(
            title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="15%", pos_left="2%")
        )
        return setup_chart(pie)
    
    # 使用更安全的饼图创建方式
    pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
    
    # 添加数据
    data_pair = []
    for genre, count in genre_counts.items():
        data_pair.append(opts.PieItem(name=genre, value=count))
    
    pie.add(
        series_name="",
        data_pair=data_pair,
        radius=["30%", "70%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
    )
    
    # 设置全局选项
    pie.set_global_opts(
        title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_top="15%",
            pos_left="2%"
        )
    )
    return setup_chart(pie)

# 添加所有图表到页面
page.add(
    create_overview(),
    create_top10_movies(),
    create_movies_by_country(),
    create_yearly_box_office_and_count(),
    create_company_stats(),
    create_domestic_movies_trend(),
    create_genre_yearly_table(),
    create_genre_pie()
)

# 保存为HTML文件
page.render("电影数据分析大屏.html")

print("数据大屏已生成，请查看'电影数据分析大屏.html'文件")

缺失值统计:
电影名         0
累计票房     3141
导演        175
主演        298
上映时间       45
国家及地区       5
发行公司      632
类型          7
链接          0
年份         45
dtype: int64
警告: 无法自动检测pyecharts渲染引擎类，尝试使用默认方式...
已使用备用渲染器


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:163: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:164: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:165: RuntimeWarning: invalid value encountered in double_scalars


AttributeError: 'Table' object has no attribute 'options'

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import seaborn as sns
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Pie, Page, Grid
from pyecharts.components import Table
from pyecharts.globals import ThemeType
from pyecharts.commons.utils import JsCode
import openpyxl
import re
import os
from jinja2 import Environment, FileSystemLoader, TemplateNotFound
import pyecharts.render.engine as engine
from typing import Any

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 读取数据
df = pd.read_excel('中国电影网电影_data.xlsx')

# 1. 数据预处理
# 清洗日期列，提取标准日期格式
def clean_date(date_str):
    # 使用正则表达式提取日期部分（格式如YYYY-MM-DD）
    match = re.search(r'(\d{4}-\d{1,2}-\d{1,2})', str(date_str))
    if match:
        return match.group(1)
    return np.nan

# 处理年份列，先清洗再提取年份
df['上映时间'] = df['上映时间'].apply(clean_date)
df['年份'] = pd.to_datetime(df['上映时间'], errors='coerce').dt.year

# 处理票房列，确保是数值类型
def clean_box_office(value):
    try:
        # 移除可能存在的逗号、货币符号等
        if isinstance(value, str):
            value = value.replace(',', '').replace('¥', '').replace('$', '').strip()
        return float(value)
    except (ValueError, TypeError):
        return np.nan

df['累计票房'] = df['累计票房'].apply(clean_box_office)

# 处理国家及地区列，取第一个国家作为主要国家
df['国家及地区'] = df['国家及地区'].str.split('/').str[0]

# 处理类型列，取第一个类型作为主要类型
df['类型'] = df['类型'].str.split('/').str[0]

# 检查并处理缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 删除关键列为NaN的行（无法确定年份或票房的数据）
df = df.dropna(subset=['年份', '累计票房'])
df['年份'] = df['年份'].astype(int)

# 2. 修改pyecharts渲染引擎以正确处理布尔值
def custom_render_chart_to_template(self, template_name: str, chart: Any, **kwargs) -> str:
    """自定义渲染函数，处理布尔值"""
    try:
        tpl = self.env.get_template(template_name)
    except TemplateNotFound:
        # 尝试从pyecharts默认模板目录加载
        pyecharts_path = os.path.dirname(engine.__file__)
        templates_path = os.path.join(pyecharts_path, 'templates')
        loader = FileSystemLoader(templates_path)
        env = Environment(loader=loader)
        
        # 添加布尔值过滤器
        def to_boolean(value):
            return "true" if value else "false"
        
        env.filters["to_boolean"] = to_boolean
        tpl = env.get_template(template_name)
    
    # 确保布尔值正确传递
    chart_dict = chart.dump_options_with_quotes()
    # 递归处理字典中的布尔值
    def process_booleans(obj):
        if isinstance(obj, dict):
            return {k: process_booleans(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [process_booleans(v) for v in obj]
        elif isinstance(obj, bool):
            return "true" if obj else "false"
        return obj
    
    chart_dict = process_booleans(chart_dict)
    return tpl.render(chart=chart_dict, **kwargs)

# 3. 动态适配不同版本的pyecharts渲染引擎
def patch_pyecharts_renderer():
    """动态检测并修补pyecharts渲染引擎"""
    # 尝试检测可用的渲染器类
    renderer_class = None
    
    # 尝试新版本的TemplateRenderer
    if hasattr(engine, 'TemplateRenderer'):
        renderer_class = engine.TemplateRenderer
    # 尝试旧版本的Engine
    elif hasattr(engine, 'Engine'):
        renderer_class = engine.Engine
    # 尝试其他可能的类名
    elif hasattr(engine, '_EChartsRenderer'):
        renderer_class = engine._EChartsRenderer
    
    if renderer_class:
        # 替换渲染方法
        renderer_class.render_chart_to_template = custom_render_chart_to_template
        print(f"已成功修补pyecharts渲染引擎: {renderer_class.__name__}")
    else:
        # 无法找到合适的渲染器类，使用备用方案
        print("警告: 无法自动检测pyecharts渲染引擎类，尝试使用默认方式...")
        # 创建一个基本的渲染器
        class BasicRenderer:
            def __init__(self):
                pyecharts_path = os.path.dirname(engine.__file__)
                templates_path = os.path.join(pyecharts_path, 'templates')
                self.env = Environment(loader=FileSystemLoader(templates_path))
            
            def render_chart_to_template(self, template_name, chart, **kwargs):
                return custom_render_chart_to_template(self, template_name, chart, **kwargs)
        
        # 使用基本渲染器
        page._renderer = BasicRenderer()
        print("已使用备用渲染器")

# 应用渲染引擎补丁
patch_pyecharts_renderer()

# 4. 创建可视化大屏
page = Page(layout=Page.DraggablePageLayout, page_title="电影数据分析大屏")

# 辅助函数：设置图表通用属性 - 完全重构以兼容不同组件类型
def setup_chart(chart, remove_br=False, page_interval=0):
    from pyecharts.charts.base import Base
    from pyecharts.components import Table
    
    if isinstance(chart, Table):
        # 处理Table组件
        try:
            # 尝试获取已有的标题
            title_opts = None
            if hasattr(chart, '_option') and 'title' in chart._option:
                title_opts = opts.ComponentTitleOpts(
                    title=chart._option['title'].get('text', ''),
                    subtitle=chart._option['title'].get('subtitle', '')
                )
            
            # 设置表格选项
            chart.set_global_opts(
                toolbox_opts=opts.ToolboxOpts(is_show=True),
                title_opts=title_opts
            )
        except Exception as e:
            print(f"警告: 设置Table选项时出错: {e}")
    elif isinstance(chart, Base):
        # 处理常规图表组件
        chart.options["remove_br"] = remove_br
        chart.options["page_interval"] = page_interval
    else:
        print(f"警告: 未知组件类型 {type(chart)}, 无法设置选项")
    
    return chart

# (1) 中一：业绩总览
def create_overview():
    # 计算累计票房
    total_box_office = df['累计票房'].sum() / 10000  # 转换为亿元
    
    # 计算各年度票房占比
    box_office_by_year = df.groupby('年份')['累计票房'].sum()
    box_office_2018 = box_office_by_year.get(2018, 0) / box_office_by_year.sum() * 100
    box_office_2017 = box_office_by_year.get(2017, 0) / box_office_by_year.sum() * 100
    box_office_2016 = box_office_by_year.get(2016, 0) / box_office_by_year.sum() * 100
    
    # 创建卡片式展示
    overview = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width="100%", height="100%"))
        .add_xaxis(["业绩总览"])
        .add_yaxis("累计票房收入(亿元)", [total_box_office], 
                  label_opts=opts.LabelOpts(position="inside", formatter="{c}"))
        .set_global_opts(
            title_opts=opts.TitleOpts(title="业绩总览", subtitle=f"2018: {box_office_2018:.1f}% | 2017: {box_office_2017:.1f}% | 2016: {box_office_2016:.1f}%"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(is_show=False)),
            tooltip_opts=opts.TooltipOpts(is_show=False),
            graphic_opts=[
                opts.GraphicText(
                    graphic_item=opts.GraphicItem(
                        left="center", top="40%"
                    ),
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text=f"{total_box_office:.2f}亿",
                        font="bold 36px Microsoft YaHei",
                        graphic_basicstyle_opts=opts.GraphicBasicStyleOpts(
                            fill="#fff"
                        )
                    )
                )
            ]
        )
    )
    return setup_chart(overview)

# (2) 左一：票房前十影片排行榜
def create_top10_movies():
    top10 = df.sort_values('累计票房', ascending=False).head(10)
    
    table = Table()
    headers = ["排名", "电影名", "票房(亿元)"]
    rows = []
    for i, (_, row) in enumerate(top10.iterrows(), 1):
        rows.append([str(i), row['电影名'], f"{row['累计票房']/10000:.2f}"])
    
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="票房前十影片排行榜")
    )
    return setup_chart(table)

# (3) 左二：各个国家影片数量的柱状图对比分析
def create_movies_by_country():
    country_counts = df['国家及地区'].value_counts().head(10)
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(country_counts.index.tolist())
        .add_yaxis("影片数量", country_counts.values.tolist())
        .set_global_opts(
            title_opts=opts.TitleOpts(title="各国影片数量TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="数量"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    return setup_chart(bar)

# (4) 中二：各个年度影片票房总额和影片数量（柱状+线性图表）
def create_yearly_box_office_and_count():
    yearly_stats = df.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_stats['累计票房'] = yearly_stats['累计票房'] / 10000  # 转换为亿元
    
    bar = (
        Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_stats['累计票房'].tolist(), yaxis_index=0)
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="年度票房与影片数量"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line = (
        Line()
        .add_xaxis(yearly_stats['年份'].tolist())
        .add_yaxis("影片数量", yearly_stats['电影名'].tolist(), yaxis_index=1)
    )
    
    bar.overlap(line)
    return setup_chart(bar)

# (5) 中三：各个电影公司的影片数量和票房总额（两个线性）
def create_company_stats():
    # 处理制作公司数据，取第一个公司作为主要公司
    df['发行公司'] = df['发行公司'].str.split('/').str[0]
    company_stats = df.groupby('发行公司').agg({'累计票房': 'sum', '电影名': 'count'}).sort_values('累计票房', ascending=False).head(10)
    company_stats['累计票房'] = company_stats['累计票房'] / 10000  # 转换为亿元
    
    line1 = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("票房总额(亿元)", company_stats['累计票房'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=company_stats['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="电影公司TOP10"),
            xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross")
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(company_stats.index.tolist())
        .add_yaxis("影片数量", company_stats['电影名'].tolist(), 
                  label_opts=opts.LabelOpts(is_show=False),
                  yaxis_index=1,
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line1.overlap(line2)
    return setup_chart(line1)

# (6) 右一：纯国产电影各个年度的票房总额和数量走势
def create_domestic_movies_trend():
    domestic = df[df['国家及地区'] == '中国']
    yearly_domestic = domestic.groupby('年份').agg({'累计票房': 'sum', '电影名': 'count'}).reset_index()
    yearly_domestic['累计票房'] = yearly_domestic['累计票房'] / 10000  # 转换为亿元
    
    line = (
        Line(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("票房总额(亿元)", yearly_domestic['累计票房'].tolist(), 
                  yaxis_index=0,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3))
        .extend_axis(
            yaxis=opts.AxisOpts(
                name="影片数量",
                type_="value",
                min_=0,
                max_=yearly_domestic['电影名'].max() * 1.2,
                axislabel_opts=opts.LabelOpts(formatter="{value}"),
                splitline_opts=opts.SplitLineOpts(is_show=True)
            )
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title="国产电影年度趋势"),
            yaxis_opts=opts.AxisOpts(name="票房(亿元)"),
            tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
            datazoom_opts=[opts.DataZoomOpts()]
        )
    )
    
    line2 = (
        Line()
        .add_xaxis(yearly_domestic['年份'].tolist())
        .add_yaxis("影片数量", yearly_domestic['电影名'].tolist(), 
                  yaxis_index=1,
                  label_opts=opts.LabelOpts(is_show=False),
                  linestyle_opts=opts.LineStyleOpts(width=3, type_="dashed"))
    )
    
    line.overlap(line2)
    return setup_chart(line)

# (7) 右二：各个年度各个影片类型的票房数据表
def create_genre_yearly_table():
    genre_yearly = df.groupby(['年份', '类型'])['累计票房'].sum().unstack().fillna(0)
    genre_yearly = genre_yearly / 10000  # 转换为亿元
    
    # 准备表格数据
    headers = ["年份"] + genre_yearly.columns.tolist()
    rows = []
    for year in genre_yearly.index:
        row = [str(year)] + [f"{v:.2f}" for v in genre_yearly.loc[year].values]
        rows.append(row)
    
    table = Table()
    table.add(headers, rows)
    table.set_global_opts(
        title_opts=opts.ComponentTitleOpts(title="年度-类型票房数据(亿元)")
    )
    return setup_chart(table)

# (8) 右三：各个类型的电影数量饼图对比分析 - 修复空数据问题
def create_genre_pie():
    genre_counts = df['类型'].value_counts().head(10)
    
    # 确保有足够的数据
    if len(genre_counts) == 0:
        # 创建空饼图作为后备 - 使用虚拟数据
        pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
        pie.add(
            "",
            [("无数据", 1)],
            radius=["30%", "70%"],
            center=["50%", "50%"],
            label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
        )
        pie.set_global_opts(
            title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_top="15%", pos_left="2%")
        )
        return setup_chart(pie)
    
    # 使用更安全的饼图创建方式
    pie = Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK))
    
    # 添加数据
    data_pair = []
    for genre, count in genre_counts.items():
        data_pair.append(opts.PieItem(name=genre, value=count))
    
    pie.add(
        series_name="",
        data_pair=data_pair,
        radius=["30%", "70%"],
        center=["50%", "50%"],
        label_opts=opts.LabelOpts(formatter="{b}: {c} ({d}%)")
    )
    
    # 设置全局选项
    pie.set_global_opts(
        title_opts=opts.TitleOpts(title="电影类型分布TOP10"),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_top="15%",
            pos_left="2%"
        )
    )
    return setup_chart(pie)

# 添加所有图表到页面
page.add(
    create_overview(),
    create_top10_movies(),
    create_movies_by_country(),
    create_yearly_box_office_and_count(),
    create_company_stats(),
    create_domestic_movies_trend(),
    create_genre_yearly_table(),
    create_genre_pie()
)

# 保存为HTML文件
page.render("电影数据分析大屏.html")

print("数据大屏已生成，请查看'电影数据分析大屏.html'文件")
def custom_render_chart_to_template(self, template_name: str, chart: Any, **kwargs) -> str:
    """自定义渲染函数，处理布尔值"""
    try:
        tpl = self.env.get_template(template_name)
    except TemplateNotFound:
        # 尝试从pyecharts默认模板目录加载
        pyecharts_path = os.path.dirname(engine.__file__)
        templates_path = os.path.join(pyecharts_path, 'templates')
        loader = FileSystemLoader(templates_path)
        env = Environment(loader=loader)
        
        # 添加布尔值过滤器 - 修正为正确的布尔值处理方式
        def to_boolean(value):
            return value
        
        env.filters["to_boolean"] = to_boolean
        tpl = env.get_template(template_name)
    
    # 确保布尔值正确传递
    chart_dict = chart.dump_options_with_quotes()
    
    # 递归处理字典中的布尔值，确保Jinja2正确解析
    def process_booleans(obj):
        if isinstance(obj, dict):
            return {k: process_booleans(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [process_booleans(v) for v in obj]
        elif isinstance(obj, bool):
            return obj  # 直接返回Python布尔值，让Jinja2正确处理
        return obj
    
    chart_dict = process_booleans(chart_dict)
    
    # 渲染模板，修正布尔值判断方式
    rendered = tpl.render(chart=chart_dict, **kwargs)
    
    # 替换模板中的布尔值判断，确保正确解析
    rendered = rendered.replace('is false', 'is not true')
    rendered = rendered.replace('is true', 'is true')  # 保持true判断不变
    
    return rendered

缺失值统计:
电影名         0
累计票房     3141
导演        175
主演        298
上映时间       45
国家及地区       5
发行公司      632
类型          7
链接          0
年份         45
dtype: int64
警告: 无法自动检测pyecharts渲染引擎类，尝试使用默认方式...
已使用备用渲染器
警告: 设置Table选项时出错: set_global_opts() got an unexpected keyword argument 'toolbox_opts'
警告: 设置Table选项时出错: set_global_opts() got an unexpected keyword argument 'toolbox_opts'


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:180: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:181: RuntimeWarning: invalid value encountered in double_scalars
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:182: RuntimeWarning: invalid value encountered in double_scalars


TemplateAssertionError: no test named 'false'